# AAI6610 Capstone: Engagement Prediction, Companion Notebook

This notebook is a clean top-to-bottom assembly of the committed model pipeline. It runs
**no new analysis**: every number and figure below comes from calling the functions in
`model/src/` (`load_data.py`, `eda.py`, `features.py`, `train.py`), and the final cell
asserts that the model metrics it computes match `docs/modeling/results.csv`. The full
write-ups live in `docs/eda_findings.md` and `docs/modeling_findings.md`.

**Dataset.** KDD Cup 1998 direct-mail set (UCI Machine Learning Repository, CC BY 4.0).
Citation: Parsa, I. (1998). KDD Cup 1998 [Data set]. https://doi.org/10.24432/C5401H .
Obtain it with `bash model/download_data.sh`, which writes `data/cup98lrn.txt` (95,412
rows by 481 columns, gitignored and never committed). This notebook reads it only through
`load_data.load_raw()`. Per the dataset terms, the sponsor is referred to as a national
veterans nonprofit; the data is used here as a public proxy for constituent engagement.

**Disclaimer.** This is an academic project for the AAI6610 course, built around the
Sturge-Weber Foundation scenario. It is **not affiliated with or endorsed by** the
Foundation, which provides no data.

**Conventions (from `CLAUDE.md`).** 80/20 stratified split, `random_state=2026`; a 15%
validation carve-out of the training portion; the imbalanced target (5.08% positive) is
judged by AUPRC and recall, never accuracy alone. The test split is read exactly once, at
the end.

**Environment.** Run this under the pinned versions in `requirements.txt` (scikit-learn
1.9.0, numpy 2.5.1, scipy 1.18.0), which is what the committed `docs/modeling/results.csv`
was generated with; the repo's `.venv` has them. The pipeline is deterministic, so under
those versions the numbers below reproduce the committed results exactly, and the final
cell asserts it. Model probabilities, and therefore the chosen threshold and its
downstream revenue and recall, do shift on other scikit-learn minor versions.

In [ ]:
# Setup: put model/src on the path and redirect figure/table output to a temp directory
# so running this notebook never overwrites anything under docs/. All data access goes
# through load_data; nothing here reads the CSV directly.
import sys, tempfile
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "CLAUDE.md").exists() and (d / "model" / "src").exists():
            return d
    raise RuntimeError("repo root not found (need CLAUDE.md and model/src above cwd)")


REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "model" / "src"))

import json
import numpy as np
import pandas as pd
from IPython.display import Image, display

import eda  # module handle, so we can redirect its output paths

# Scratch dir for any figure or table a function writes as a side effect. Keeping it out
# of the repo means the notebook is reproducible without dirtying committed docs/.
NB_OUT = Path(tempfile.mkdtemp(prefix="capstone_nb_"))
eda.FIGURES_DIR = NB_OUT
eda.EDA_DIR = NB_OUT
eda.INVENTORY_PATH = NB_OUT / "column_inventory.csv"

print("repo root:", REPO_ROOT)
print("scratch output dir:", NB_OUT)

In [ ]:
from load_data import load_raw, make_split, RANDOM_STATE, TARGET_BINARY, TARGET_AMOUNT
from features import make_xy

# The frames, and which stage may see each (see train.py). The test split stays untouched
# until the final evaluation cell.
df = load_raw()
train_full, test = make_split(df)                      # 76,329 / 19,083
train_inner, validation, test_check = make_split(df, validation=True)  # 64,879 / 11,450 / 19,083
assert test.index.equals(test_check.index), "the two split calls must agree on test"

print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns  (random_state={RANDOM_STATE})")
for name, frame in [("train_full", train_full), ("train_inner", train_inner),
                    ("validation", validation), ("test", test)]:
    print(f"  {name:<11} {len(frame):>6,} rows  ({frame[TARGET_BINARY].mean()*100:.2f}% positive)")

## 1. Exploratory data analysis (condensed)

Called from `model/src/eda.py`. The peeking rule holds throughout: structural facts use
the full dataset, anything touching the target uses the training split only. This is a
condensed pass (the column inventory, the target and giving-history headline numbers, and
five headline figures); the full six-task EDA is in `docs/eda_findings.md`.

In [ ]:
from eda import apply_style, task1_inventory, task2_targets, task5_relationships

apply_style()

# Task 1: the 481-row column inventory (structural facts, full dataset). Returns the table.
inventory = task1_inventory(df)
print("Column inventory:", inventory.shape[0], "columns")
print("\nColumns per group (census dominates the feature space):")
print(inventory["group"].value_counts().to_string())
inventory.head(8)

In [ ]:
# Task 2: both targets, on the training split. Prints the headline numbers and writes the
# TARGET_D histogram (to the scratch dir).
task2_targets(train_full, df)

In [ ]:
# Task 5: response rate by segment and correlations with response, on the training split.
# Prints the RFA / INCOME / AGE tables and writes the response-rate and correlation figures.
task5_relationships(train_full)

In [ ]:
# Five headline figures, each just generated by the eda.py functions above.
headline_figures = [
    ("target_d_distribution_responders.png", "Responder donations cluster on round amounts ($10 mode)"),
    ("response_rate_by_rfa2a.png", "Response falls as past gift size rises (2.75x, strongest signal)"),
    ("response_rate_by_rfa2f.png", "Response rises with giving frequency (2.21x)"),
    ("response_rate_by_income.png", "Response rises gently with income bracket"),
    ("correlation_giving_history.png", "Giving-history features correlate with each other, not with response"),
]
for fname, caption in headline_figures:
    print(caption)
    display(Image(str(NB_OUT / fname)))

## 2. Model comparison

Called from `model/src/train.py`. Three model families plus a dummy baseline, tuned by
stratified 5-fold cross-validation on the training portion, scored by AUPRC (average
precision). The dummy anchors AUPRC at the 5.08% positive rate; a model that cannot clear
it has found nothing. This cell fits the full grid, so it takes a few minutes.

In [ ]:
from train import (
    AUPRC_FLOOR, RESULTS_PATH, model_specs, run_cv, fit_spec,
    figure_cv_comparison,
)

# Tune every model by CV on train_full. Same code path for every row, including the dummy.
rows = []
for spec in model_specs():
    print(f"Fitting {spec.name} ...", flush=True)
    rows.append(run_cv(spec, *make_xy(train_full)[:2]))

cv_table = pd.DataFrame(
    [
        {
            "model": r["model"],
            "cv_auprc_mean": round(r["cv_auprc_mean"], 4),
            "cv_auprc_std": round(r["cv_auprc_std"], 4),
            "lift_vs_floor": round(r["cv_auprc_mean"] / AUPRC_FLOOR, 2),
            "params": r["params"],
        }
        for r in rows
    ]
)
print(f"\nAUPRC floor (random baseline = positive rate): {AUPRC_FLOOR:.4f}")
cv_table

In [ ]:
figure_cv_comparison(rows)
display(Image(str(NB_OUT / "cv_auprc_by_model.png")))

### Threshold selection and the net-revenue lens

Response rate and gift size pull in opposite directions (the EDA above), so a model
thresholded on response probability alone prefers the constituents who give least. The
resolution is to bring the amounts back at threshold selection, through net revenue: the
TARGET_D of contacted responders minus the $0.68 mail cost per contact. Models are refitted
on `train_inner` here so the validation rows they are scored on are genuinely held out.

In [ ]:
from train import revenue_curve, net_revenue, choose_threshold, figure_net_revenue, figure_calibration

X_inner, y_inner, _ = make_xy(train_inner)
X_val, y_val, amounts_val = make_xy(validation)
y_val_a, amounts_val_a = y_val.to_numpy(), amounts_val.to_numpy()

curves, probabilities = {}, {}
for r, spec in zip(rows, model_specs()):
    est = fit_spec(spec, json.loads(r["params"]), X_inner, y_inner)
    p = est.predict_proba(X_val)[:, 1]
    probabilities[spec.name] = p
    curves[spec.name] = revenue_curve(y_val_a, amounts_val_a, p)

everyone = net_revenue(y_val_a, amounts_val_a, np.ones(len(y_val_a), dtype=bool))
best_name = max(rows, key=lambda r: r["cv_auprc_mean"])["model"]
chosen = choose_threshold(y_val_a, amounts_val_a, probabilities[best_name])

print(f"Best model by CV AUPRC: {best_name}")
print(f"Mailing everyone on validation nets ${everyone:,.2f}")
print(f"Chosen threshold {chosen['threshold']:.4f}: {chosen['contacts']:,} contacts "
      f"({chosen['contact_fraction']*100:.1f}%), recall {chosen['recall']*100:.1f}%, "
      f"net ${chosen['net_revenue']:,.2f}")

figure_net_revenue(curves, everyone)
display(Image(str(NB_OUT / "net_revenue_curve.png")))

In [ ]:
# Reliability of the best model's probabilities on the validation carve-out.
figure_calibration(best_name, y_val_a, probabilities[best_name])
display(Image(str(NB_OUT / "calibration_best_model.png")))

## 3. Fairness slices

A proposal commitment and part of the graded ethics content. Measurement only: contact
rate and recall by demographic segment on the validation carve-out, at the chosen
threshold. **These scores inform outreach efficiency only. They must never gate any
family's access to services** (`CLAUDE.md`). A contact-rate gap means one group gets less
mail; it must never come to mean one group gets less help.

In [ ]:
from train import fairness_slices, figure_fairness

slices = fairness_slices(validation, probabilities[best_name], chosen["threshold"])
figure_fairness(slices, chosen["threshold"])

# Contact-rate gap across the segments large enough to read (n >= 100), per attribute.
for attribute, group in slices.groupby("attribute", sort=False):
    big = group[group["n"] >= 100]
    if len(big) > 1:
        gap = (big["contact_rate"].max() - big["contact_rate"].min()) * 100
        hi = big.loc[big["contact_rate"].idxmax(), "segment"]
        lo = big.loc[big["contact_rate"].idxmin(), "segment"]
        print(f"{attribute:<10} contact-rate gap {gap:4.1f} pp  ({hi} highest, {lo} lowest)")

display(slices.round(4))
display(Image(str(NB_OUT / "fairness_slices.png")))

## 4. Test evaluation (read once)

The model family, its hyperparameters, and the operating threshold are now frozen. This is
the only cell that reads the test split. The model is refitted on the full training portion
(what a campaign would ship), then evaluated once at the frozen threshold.

In [ ]:
from train import evaluate_on_test

best_spec = next(s for s in model_specs() if s.name == best_name)
best_params = json.loads(next(r for r in rows if r["model"] == best_name)["params"])
test_result = evaluate_on_test(
    best_spec, best_params, chosen["threshold"], *make_xy(train_full)[:2], test
)

## 5. Every number matches the committed results

Final check: the metrics this notebook computed by executing the pipeline match
`docs/modeling/results.csv`, the committed table the findings doc quotes. The pipeline is
deterministic (seed 2026), so these agree exactly.

In [ ]:
committed = pd.read_csv(RESULTS_PATH).set_index("model")

# CV AUPRC per model, computed here vs committed.
for r in rows:
    expected = committed.loc[r["model"], "cv_auprc_mean"]
    assert np.isclose(r["cv_auprc_mean"], expected, atol=1e-6), \
        f"{r['model']}: {r['cv_auprc_mean']} != committed {expected}"

# The winning model and its frozen test metrics.
best = committed.loc[best_name]
assert best_name == "hist_gradient_boosting", best_name
assert np.isclose(round(test_result["auprc"], 4), best["test_auprc"], atol=1e-6), \
    (test_result["auprc"], best["test_auprc"])
assert np.isclose(round(test_result["net_revenue"], 2), best["test_net_revenue"], atol=0.01), \
    (test_result["net_revenue"], best["test_net_revenue"])
assert np.isclose(round(chosen["threshold"], 6), best["test_threshold"], atol=1e-6)

print("All notebook metrics match docs/modeling/results.csv")
print(f"  best model:     {best_name}")
print(f"  test AUPRC:     {test_result['auprc']:.4f}  (floor {test_result['positive_rate']:.4f}, "
      f"lift {test_result['auprc']/test_result['positive_rate']:.2f}x)")
print(f"  test recall:    {test_result['recall']*100:.2f}% at threshold {chosen['threshold']:.4f}")
print(f"  test net rev:   ${test_result['net_revenue']:,.2f} "
      f"vs ${test_result['everyone']:,.2f} mailing everyone "
      f"({(test_result['net_revenue']/test_result['everyone']-1)*100:+.1f}%)")

## Where to read more

- Full EDA (six tasks, all figures and tables): `docs/eda_findings.md`, `docs/eda/`.
- Full modeling write-up (calibration, the census experiment, the fairness discussion, the
  net-revenue economics): `docs/modeling_findings.md`, `docs/modeling/`.
- The family navigation assistant (the project's second build): `docs/assistant_findings.md`.
- Strategic AI roadmap across three horizons: `docs/roadmap.md`.

Nothing in this notebook is new analysis. It is a single reproducible pass over the
committed pipeline, and the assertions above tie its numbers to the committed results.